In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
!pip install -q -U transformers huggingface_hub accelerate "bitsandbytes>=0.46.1"

In [2]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge/"
test_df = pd.read_csv(BASE + "test.csv")
sample_sub = pd.read_csv(BASE + "sample_submission.csv")
options = ["A", "B", "C", "D", "E"]


test: (500, 7) | columns needed: ['ID', 'Prediction']


In [5]:
PREFIXES = ["Pick the best possible answer:", "Select the most accurate option:",
            "Identify the correct statement:", "Determine the correct option:",
            "Choose the correct answer:", "Which of the following is correct?"]
SUFFIXES = ["among the listed options.", "from the following choices.",
            "carefully.", "based on the given context.", "among the list of options."]


def clean_prompt(text):
    text = str(text).strip()
    for p in PREFIXES:
        if text.startswith(p):
            text = text[len(p):].strip(); break
    for s in SUFFIXES:
        if text.endswith(s):
            text = text[:-len(s)].strip(); break
    return text


test_df["question"] = test_df["prompt"].apply(clean_prompt)
print("filler removed from", (test_df["question"] != test_df["prompt"].str.strip()).sum(), "rows")

filler removed from 300 rows


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"   

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model.eval()
print("model loaded")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded


In [6]:
SYSTEM = ("You are an expert in physics, astronomy, chemistry, engineering and "
          "statistics. You answer multiple choice questions correctly.")

PREFILL = "Answer:"


def build_text(row, perm):
    lines = ["Question: " + str(row["question"]), ""]
    for slot in range(5):
        lines.append(options[slot] + ". " + str(row[options[perm[slot]]]).strip())
    lines.append("")
    lines.append("Reply with one letter only: A, B, C, D, or E.")
    return "\n".join(lines)


def build_prompt(row, perm):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": build_text(row, perm)}]
    base = tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)
    return base + PREFILL


# check the rotation moves the options correctly
demo = build_text(test_df.iloc[0], [2, 3, 4, 0, 1])
line_a = [l for l in demo.split("\n") if l.startswith("A. ")][0]
print("rotation works:", str(test_df.iloc[0]["C"]).strip()[:40] in line_a)
print()
print(build_prompt(test_df.iloc[0], [0,1,2,3,4])[-320:])

rotation works: True

ry eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a different energy.
E. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a lower energy.

Reply with one letter only: A, B, C, D, or E.<|im_end|>
<|im_start|>assistant
Answer:


In [8]:
N_PERMS = 5


def get_option_scores(row):
    perms, prompts = [], []
    for r in range(N_PERMS):
        perm = [(slot + r) % 5 for slot in range(5)]
        perms.append(perm)
        prompts.append(build_prompt(row, perm))

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model(**inputs)

    letter_logits = out.logits[:, -1, :][:, letter_id_list]
    log_probs = torch.log_softmax(letter_logits.float(), dim=-1).cpu().numpy()

    totals = np.zeros(5)
    for r in range(N_PERMS):
        for slot in range(5):
            totals[perms[r][slot]] += log_probs[r, slot]
    return totals / N_PERMS


def rank_from_scores(s):
    return [options[i] for i in np.argsort(-s)]


for i in range(2):
    s = get_option_scores(test_df.iloc[i])
    print(test_df.iloc[i]["question"][:70])
    print("  ", np.round(s, 3), "->", rank_from_scores(s))

What is the relationship between the Hamiltonians and eigenstates in s
   [ -3.228 -13.603 -20.303  -5.878 -10.803] -> ['A', 'D', 'E', 'B', 'C']
What is the estimated redshift of CEERS-93316, a candidate high-redshi
   [-12.271  -0.621 -18.121  -9.046  -9.221] -> ['B', 'D', 'E', 'A', 'C']


In [9]:
test_scores, test_ranked = [], []

for i in tqdm(range(len(test_df))):
    s = get_option_scores(test_df.iloc[i])
    test_scores.append(s)
    test_ranked.append(rank_from_scores(s))

np.save("test_scores_32b_prefill.npy", np.array(test_scores))
print("scores saved")

  0%|          | 0/500 [00:00<?, ?it/s]

scores saved


In [10]:
submission = pd.DataFrame({
    sample_sub.columns[0]: test_df["id"],
    sample_sub.columns[1]: [" ".join(r[:3]) for r in test_ranked],
})

ok = all(len(set(p.split())) == 3 and all(l in options for l in p.split())
         for p in submission[sample_sub.columns[1]])
print("rows:", len(submission), "| valid:", ok,
      "| dup ids:", submission[sample_sub.columns[0]].duplicated().sum())

submission.to_csv("submission.csv", index=False)
print()
print("how often each letter came first:")
print(pd.Series([r[0] for r in test_ranked]).value_counts().sort_index())
submission.head()

rows: 500 | valid: True | dup ids: 0

how often each letter came first:
A     86
B    103
C    126
D     98
E     87
Name: count, dtype: int64


,ID,Prediction
0,1,A D E
1,2,B D E
2,3,C B E
3,4,E C A
4,5,C D A
